# Modelling — LightGBM
## Credit Risk Scorecard Project

This notebook builds a LightGBM gradient boosting model.
We use all 267 features (vs 18 for the scorecard) and compare
performance against the WOE scorecard from notebook 03.

LightGBM trades interpretability for raw predictive power.
The comparison between the two models is the centrepiece of this project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

print("Libraries loaded successfully")
print(f"LightGBM version: {lgb.__version__}")

In [ ]:
# Load cleaned dataset
app = pd.read_csv('data/processed/app_clean.csv')

# Split features and target
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app['TARGET']

# Same split as scorecard notebook for fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Default rate: {y_train.mean():.2%}")

In [ ]:
import re

# Clean column names - remove special characters
X_train.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X_test.columns]

print("Column names cleaned")

# Handle class imbalance with scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"Class imbalance ratio: {scale:.1f}")

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'scale_pos_weight': scale,
    'random_state': 42,
    'verbose': -1
}

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

print("Training LightGBM model...")

# Train model
model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print(f"\nBest iteration: {model.best_iteration}")

In [ ]:
# Get predictions on test set
test_pred_lgb = model.predict(X_test, num_iteration=model.best_iteration)
test_auc_lgb = roc_auc_score(y_test, test_pred_lgb)

print(f"LightGBM Test AUC: {test_auc_lgb:.4f}")
print(f"Scorecard Test AUC: 0.7391 (from notebook 03)")
print(f"Improvement: +{(test_auc_lgb - 0.7391):.4f}")

# Feature importance
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 20 most important features:")
print(importance.head(20).to_string(index=False))

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance.head(20)
ax.barh(top20['feature'], top20['importance'], color='steelblue')
ax.invert_yaxis()
ax.set_title('LightGBM Top 20 Feature Importance (by Gain)')
ax.set_xlabel('Importance (Gain)')
plt.tight_layout()
plt.savefig('outputs/04_feature_importance.png', dpi=150)
plt.show()

print("Feature importance chart saved ")


### Feature Importance Analysis

**External credit scores dominate.** EXT_SOURCE_1, 2, and 3 are by far 
the strongest predictors, together accounting for the majority of model 
gain. These are pre-computed creditworthiness scores from external 
bureaus which is effectively another institution's risk assessment. Their 
dominance is consistent with the WOE scorecard's IV ranking, confirming 
both models independently identify the same core signal.

**Engineered features validated.** Three features built during data 
preparation rank in the top 20: CREDIT_GOODS_RATIO (#4), 
EMPLOYED_TO_AGE_RATIO (#17), and ANNUITY_INCOME_RATIO (#20). This 
confirms the domain-driven feature engineering added genuine predictive 
value rather than noise.

**Behavioural history matters.** PREV_REFUSED_COUNT (#10) and multiple 
BUREAU_ aggregations appear prominently, justifying the effort to 
aggregate and join the external history tables. Past borrowing 
behaviour is among the strongest non-bureau-score signals.

**Fairness concern flagged.** CODE_GENDER_F ranks #12, meaning gender 
is a meaningful predictor. Using gender in credit decisions is 
prohibited under fair lending regulations in many jurisdictions. In 
production this feature should be removed and the model tested for 
disparate impact across protected groups.